In [2]:
import json
import re
import sys
from pathlib import Path

In [ ]:
CONCEPTS = [
    ("TotalAssetsLessCurrentLiabilities",                       r"total assets less current liabilit"),
    ("NetCurrentAssetsLiabilities",                             r"net current (assets|liabilit)"),
    ("TotalAssetsLiabilities",                                  r"(total )?net (assets|liabilit)"),
    ("Equity",                                         r"capital and reserves"),
    ("CalledUpShareCapitalNotPaidNotExpressedAsCurrentAsset",   r"called.?up share capital"),
    ("FixedAssets",                                             r"fixed assets"),
    ("CurrentAssets",                                           r"current assets"),
    ("Debtors",                                                 r"debtors"),
    ("CashBankOnHand",                                          r"cash (at bank|in hand)"),
    ("ProfitLoss",                                              r"(profit and loss account|retained earnings)"),
    ("AverageNumberEmployeesDuringPeriod",                      r"average number of employees"),
]

In [4]:
def find_concept(label):
    """Return the concept name matching a row label, or None."""
    text = str(label or "").strip().lower()
    for name, pattern in CONCEPTS:
        if re.match(pattern, text):
            return name
    return None

In [39]:
def parse_number(cell):
    """Parse a cell to a float. Brackets mean negative, a dash means nil."""
    text = str(cell or "").strip()
    if not text:
        return None
    if re.fullmatch(r"[-\u2010-\u2015\u2212]+", text):
        return 0.0
 
    negative = text.startswith("(") and text.endswith(")")
    text = re.sub(r"[£$€,()\s]", "", text)
 
    if not re.fullmatch(r"-?\d+(\.\d+)?", text):
        return None
    value = float(text)
    return -value if negative else value

def parse_year(cell):
    text = str(cell or "").strip()
    if not text:
        return None
    year = re.search(r"20\d\d", text)
    
    return int(text[year.start():year.end()]) if year else None

In [40]:
def extract(path):
    """Yield (page, concept, column, value) for one company file."""
    out_features = {}

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
 
    for page, payload in data.items():

        if payload[0].get("table_found") is False:
            continue

        for table in payload[0].get("tables") or []:
            headers = table.get("headers") or []
 
            # Roughly a quarter of tables carry a "Notes" column of reference
            # numbers. Left in, those get read as values.
            skip = {i for i, h in enumerate(headers[1:])
                    if str(h or "").strip().lower().startswith("note")}
 
            for row in table.get("rows") or []:
                if not row:
                    continue
                print(row)
                concept = find_concept(row[0])
                if not concept:
                    continue
                for i, cell in enumerate(row[1:]):
                    if i in skip:
                        continue
                    value = parse_number(cell)
                    if value is not None:
                        year = parse_year(headers[i+1])
                        if not out_features.get(year):
                            out_features[year] = {}
                        out_features[year][concept] = value

    return out_features

In [51]:
paths = [Path("C:/Users/Ink/Downloads/01612140.json")]

for path in paths:
    out_features = extract(path)

    with open(path.stem + ".json", "w", newline="") as f:
        json.dump(out_features, f, indent=2)

['Called up share capital not paid', '4', '4']
['Fixed Assets', None, None]
['Current Assets', None, None]
['Prepayments and accrued income', '1,110', '4,424']
['Creditors: amounts falling due within one year', '(1,052)', '(4,366)']
['Net current assets (liabilities)', '58', '58']
['Total assets less current liabilities', '62', '62']
['Creditors: amounts falling due after more than one year', '0', '0']
['Provisions for liabilities', '0', '0']
['Accruals and deferred income', '0', '0']
['Total net assets (liabilities)', '62', '62']
['Capital and reserves', '62', '62']
['Average number of employees during the period', '0', '0']
